## 买入并持有

策略描述：

买入并持有策略是一种经典的长期投资策略，投资者在购买股票或其他资产后，长期持有，通常不受市场短期波动的影响。该策略的核心理念是，相信优质资产的长期增长潜力，通过耐心持有来获得资本增值和股息收益，而不频繁进行买卖操作。这种策略适合相信市场整体向上趋势的投资者，并能避免因短期市场波动而做出的情绪化决策。

在这个策略中，投资者会选择一些具有长期增长潜力的股票。例如，本策略持有300059.SZ（东方财富）和600519.SH（贵州茅台）都是A股具有代表性的股票。东方财富作为金融科技龙头，长期受益于中国金融市场的数字化发展；而贵州茅台作为中国领先的白酒企业，具有品牌优势和强大的市场需求，长期以来股价表现稳健。这两只股票具有良好的基本面和市场前景，符合买入并持有策略的长期价值投资理念。

In [ ]:
from bigmodule import M, I

# <aistudiograph>

# @param(id="m5", name="initialize")
# 交易引擎：初始化函数，只执行一次
def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder

    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))

# @param(id="m5", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m5_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m5", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m5_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m5", name="handle_data")
def m5_handle_data_bigquant_run(context, data):
    import pandas as pd

    # 下一个交易日不是调仓日，则不生成信号
    if not context.rebalance_period.is_signal_date(data.current_dt.date()):
        return
    
 
    # 从传入的数据 context.data 中读取今天的信号数据
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])

    # 获取当前已持有股票
    holding_instruments = set(context.get_account_positions().keys())

    # 卖出不在目标持有列表中的股票
    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)
    # 买入目标持有列表中的股票
    for i, x in today_df.iterrows():
        # 处理 null 或者 decimal.Decimal 类型等
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)

# @param(id="m5", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m5_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m5", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m5_handle_order_bigquant_run(context, order):
    pass

# @param(id="m5", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m5_after_trading_bigquant_run(context, data):
    pass

# @module(position="-311,-736", comment="""因子特征""")
m2 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""close as score""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
instrument in ('300059.SZ', '600519.SH')


""",
    expr_tables="""cn_stock_prefactors_community""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据，构建因子等，如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B

SELECT

    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_factors https://bigquant.com/data/datasources/cn_stock_factors
    cn_stock_factors
WHERE
    -- WHERE 过滤，在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤，在窗口等计算算子之后执行，比如 m_lag(close, 3) AS close_3，对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
-- 按日期和股票代码排序，从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m2"""
)

# @module(position="-312,-630", comment="""持股数量、打分到仓位""")
m3 = M.score_to_position.v4(
    input_1=m2.data,
    score_field="""score asc""",
    hold_count=2,
    position_expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 在这里输入表达式, 每行一个表达式, 输出仓位字段必须命名为 position, 模块会进一步做归一化
-- 排序倒数: 1 / score_rank AS position
-- 对数下降: 1 / log2(score_rank + 1) AS position
-- TODO 拟合、最优化 ..

-- 等权重分配
1 AS position
""",
    total_position=1,
    extract_data=False,
    m_name="""m3"""
)

# @module(position="-307,-521", comment="""抽取预测数据""")
m4 = M.extract_data_dai.v17(
    sql=m3.data,
    start_date="""2019-01-01""",
    start_date_bound_to_trading_date=True,
    end_date="""2024-09-11""",
    end_date_bound_to_trading_date=True,
    before_start_days=90,
    debug=False,
    m_name="""m4"""
)

# @module(position="-304,-406", comment="""交易，日线，设置初始化函数和K线处理函数，以及初始资金、基准等""")
m5 = M.bigtrader.v34(
    data=m4.data,
    start_date="""""",
    end_date="""""",
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""126""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""open""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m5"""
)
# </aistudiograph>

[2024-11-27 17:09:30] [info     ] input_features_dai.v30 开始运行 ..
[2024-11-27 17:09:30] [info     ] expr mode
[2024-11-27 17:09:30] [info     ] input_features_dai.v30 运行完成 [0.037s].
[2024-11-27 17:09:30] [info     ] score_to_position.v4 开始运行 ..
[2024-11-27 17:09:30] [info     ] score_to_position.v4 运行完成 [0.294s].
[2024-11-27 17:09:30] [info     ] extract_data_dai.v17 开始运行 ..


[2024-11-27 17:09:32] [info     ] data extracted: (2768, 5)
[2024-11-27 17:09:32] [info     ] extract_data_dai.v17 运行完成 [1.607s].
[2024-11-27 17:09:32] [info     ] bigtrader.v34 开始运行 ..
[2024-11-27 17:09:32] [info     ] got metadata extra from input datasource
[2024-11-27 17:09:32] [info     ] read input 'data' ..
[2024-11-27 17:09:32] [info     ] 2019-01-01, 2024-09-11, , equity, instruments=2
[2024-11-27 17:09:32] [info     ] bigtrader module V2.2.0
[2024-11-27 17:09:32] [info     ] bigtrader engine v1.10.10 2024-11-21


INFO:MAIN:======== bigtrader pid:11999 version 1.10.10 2024-11-21 ========
INFO:MAIN:bigtrader run_mode:BACKTEST, handle_bar_mode:0, frequency:1d, exchange_mode:BQ2
INFO:MAIN:> process add account:BACKTEST,0,bkt000
INFO:ACCT[bkt000]:AccountEngine: self_calc:1, validate_self_trading:0, validate_cash:0, validate_position:1, enable_auto_planed_order:1
INFO:MAIN:login_account(bkt000)
INFO:MAIN:> add_strategy setting:{'strategy_name': 'strategy', 'account_id': 'bkt000'}
INFO:MAIN:init all strategy account_id:...
INFO:MAIN:stop all strategy account_id:...


[2024-11-27 17:09:37] [info     ] backtest done, raw_perf_ds:dai.DataSource("_7820862b5a4c47efa1abd0e662273f24")


[2024-11-27 17:09:40] [info     ] bigtrader.v34 运行完成 [8.249s].
